In [44]:
import pandas as pd

df = pd.read_csv("Global_Mobility_Report.csv", low_memory=False)
df.head()

,country_region_code,country_region,sub_region_1,sub_region_2,metro_area,iso_3166_2_code,census_fips_code,place_id,date,retail_and_recreation_percent_change_from_baseline,grocery_and_pharmacy_percent_change_from_baseline,parks_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline
0,AE,United Arab Emirates,NaN,NaN,NaN,NaN,NaN,ChIJvRKrsd9IXj4RpwoIwFYv0zM,2020-02-15,0.0,4.0,5.0,0.0,2.0,1.0
1,AE,United Arab Emirates,NaN,NaN,NaN,NaN,NaN,ChIJvRKrsd9IXj4RpwoIwFYv0zM,2020-02-16,1.0,4.0,4.0,1.0,2.0,1.0
2,AE,United Arab Emirates,NaN,NaN,NaN,NaN,NaN,ChIJvRKrsd9IXj4RpwoIwFYv0zM,2020-02-17,-1.0,1.0,5.0,1.0,2.0,1.0
3,AE,United Arab Emirates,NaN,NaN,NaN,NaN,NaN,ChIJvRKrsd9IXj4RpwoIwFYv0zM,2020-02-18,-2.0,1.0,5.0,0.0,2.0,1.0
4,AE,United Arab Emirates,NaN,NaN,NaN,NaN,NaN,ChIJvRKrsd9IXj4RpwoIwFYv0zM,2020-02-19,-2.0,0.0,4.0,-1.0,2.0,1.0


Fase 2: Limpieza de Datos (Data Cleaning)

Carga el archivo Global_Mobility_Report.csv. Notarás que un dataset real está
"sucio":

• Filtro: Selecciona solo los datos de Chile (columna country_region).

• Nulos: Identifica columnas vacías. Decide: ¿Borras o imputas? (Justifícalo
en un comentario).

• Duplicados: Verifica registros repetidos por fecha y subregión.

• Outliers: Revisa variaciones de movilidad imposibles (ej. errores de
digitación).

In [45]:
# ============================================================
# FASE 2: Limpieza de Datos (Data Cleaning)
# ============================================================

# --- 1. Filtro: Solo datos de Chile ---
df_chile = df[df["country_region"] == "Chile"].copy()
print(f"Registros totales en el dataset: {len(df)}")
print(f"Registros de Chile: {len(df_chile)}")
print(f"\nPrimeras filas de Chile:")
df_chile.head()

Registros totales en el dataset: 11730025
Registros de Chile: 68716

Primeras filas de Chile:


,country_region_code,country_region,sub_region_1,sub_region_2,metro_area,iso_3166_2_code,census_fips_code,place_id,date,retail_and_recreation_percent_change_from_baseline,grocery_and_pharmacy_percent_change_from_baseline,parks_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline
3259335,CL,Chile,NaN,NaN,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-15,2.0,4.0,9.0,0.0,-3.0,0.0
3259336,CL,Chile,NaN,NaN,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-16,3.0,5.0,5.0,4.0,-1.0,0.0
3259337,CL,Chile,NaN,NaN,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-17,1.0,6.0,11.0,-3.0,-8.0,1.0
3259338,CL,Chile,NaN,NaN,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-18,0.0,5.0,13.0,-3.0,-7.0,1.0
3259339,CL,Chile,NaN,NaN,NaN,NaN,NaN,ChIJL68lBEHFYpYRHbkCERPhBQU,2020-02-19,0.0,8.0,11.0,-3.0,-7.0,1.0


In [ ]:
# --- 2. Nulos: Identificar columnas con valores faltantes ---

print("Valores nulos por columna:")
print(df_chile.isnull().sum())

Valores nulos por columna:


,Columna,Nulos,% Nulos
0,country_region_code,0,0.00
1,country_region,0,0.00
2,sub_region_1,974,1.42
3,sub_region_2,16556,24.09
4,metro_area,68716,100.00
5,iso_3166_2_code,53134,77.32
6,census_fips_code,68716,100.00
7,place_id,0,0.00
8,date,0,0.00
9,retail_and_recreation_percent_change_from_base...,6295,9.16


In [ ]:
# --- 2. ¿Se imputa o se elimina? ---

cols_eliminar = ['metro_area', 'census_fips_code', 'iso_3166_2_code']
cols_vacias = [col for col in cols_eliminar if col in df_chile.columns]
print(f"Columnas eliminadas: {cols_vacias}")
df_chile.drop(columns=cols_vacias, inplace=True)

print("\nTratamiento de nulos en columnas de región:")
print(f"sub_region_1: {df_chile['sub_region_1'].isnull().sum()} nulos -> se rellenan con 'Nacional'")
print(f"sub_region_2: {df_chile['sub_region_2'].isnull().sum()} nulos -> se rellenan con 'Sin especificar'")

df_chile['sub_region_1'] = df_chile['sub_region_1'].fillna('Nacional')
df_chile['sub_region_2'] = df_chile['sub_region_2'].fillna('Sin especificar')

cols_movilidad = [col for col in df_chile.columns if "percent_change" in col]

for col in cols_movilidad:
    nulos_antes = df_chile[col].isnull().sum()
    if nulos_antes > 0:
        mediana = df_chile[col].median()
        print(f"\nLa mediana de {col} es: {mediana}")
        df_chile[col] = df_chile[col].fillna(mediana)
        print(f"  -> {nulos_antes} nulos rellenados con la mediana")

print("\nValores nulos después:")
print(df_chile.isnull().sum())

Columnas eliminadas: ['metro_area', 'census_fips_code', 'iso_3166_2_code']

Tratamiento de nulos en columnas de región:
sub_region_1: 974 nulos -> se rellenan con 'Nacional'
sub_region_2: 16556 nulos -> se rellenan con 'Sin especificar'

La mediana de retail_and_recreation_percent_change_from_baseline es: -20.0
  -> 6295 nulos rellenados con la mediana

La mediana de grocery_and_pharmacy_percent_change_from_baseline es: 0.0
  -> 10520 nulos rellenados con la mediana

La mediana de parks_percent_change_from_baseline es: -33.0
  -> 186 nulos rellenados con la mediana

La mediana de transit_stations_percent_change_from_baseline es: -24.0
  -> 8522 nulos rellenados con la mediana

La mediana de workplaces_percent_change_from_baseline es: -4.0
  -> 5585 nulos rellenados con la mediana

La mediana de residential_percent_change_from_baseline es: 12.0
  -> 10662 nulos rellenados con la mediana

Valores nulos después:
country_region_code                                   0
country_region       

## Justificación del tratamiento de valores nulos

**Decisiones tomadas:**

1. **Columnas eliminadas** (`metro_area`, `census_fips_code`, `iso_3166_2_code`):
   - `metro_area` y `census_fips_code`: 100% vacías (son datos específicos de EE.UU., no aplican para Chile)
   - `iso_3166_2_code`: 77% vacía y **redundante** (la información ya está en `sub_region_1` y `sub_region_2`)

2. **Columnas imputadas** (`sub_region_1`, `sub_region_2`):
   - `sub_region_1 = NaN` → representa datos agregados a **nivel nacional** (Chile completo)
   - `sub_region_2 = NaN` → representa datos sin desglose provincial
   - **Solución**: Rellenar con categorías descriptivas (`'Nacional'` y `'Sin especificar'`) para mantener estos registros valiosos

3. **Columnas de movilidad**: Rellenadas con la **mediana** debido a que los datos tienen distribuciones no normales y la mediana es más robusta que la media ante estos outliers.

In [37]:
# --- 3. Duplicados: Verificar registros repetidos por fecha y subregión ---

duplicados = df_chile.duplicated(subset=['date', 'sub_region_1', 'sub_region_2'], keep=False)
n_duplicados = duplicados.sum()
print(f"Registros duplicados por fecha + sub_region_1 + sub_region_2: {n_duplicados}")

if n_duplicados > 0:
    print("\nEjemplos de registros duplicados:")
    print(df_chile[duplicados].sort_values(['date', 'sub_region_1', 'sub_region_2']).head(10))
    
    # Eliminar duplicados, manteniendo el primer registro
    antes = len(df_chile)
    df_chile.drop_duplicates(subset=['date', 'sub_region_1', 'sub_region_2'], keep='first', inplace=True)
    despues = len(df_chile)
    print(f"\nRegistros antes: {antes}")
    print(f"Registros después: {despues}")
    print(f"Registros eliminados: {antes - despues}")
else:
    print("No se encontraron duplicados.")

Registros duplicados por fecha + sub_region_1 + sub_region_2: 0
No se encontraron duplicados.


In [47]:
# --- 4. Outliers: Detectar variaciones de movilidad imposibles ---

cols_movilidad = [col for col in df_chile.columns if "percent_change" in col]

resumen = []
for col in cols_movilidad:
    nombre_corto = col.replace("_percent_change_from_baseline", "")
    fuera_rango = ((df_chile[col] > 200) | (df_chile[col] < -100)).sum()
    resumen.append({
        'Columna': nombre_corto,
        'Mínimo': df_chile[col].min(),
        'Máximo': df_chile[col].max(),
        'Fuera de rango': fuera_rango
    })

print("ANTES de corregir outliers:")
df_resumen = pd.DataFrame(resumen)
display(df_resumen)

for col in cols_movilidad:
    df_chile.loc[df_chile[col] > 200, col] = 200
    df_chile.loc[df_chile[col] < -100, col] = -100

resumen_despues = []
for col in cols_movilidad:
    nombre_corto = col.replace("_percent_change_from_baseline", "")
    resumen_despues.append({
        'Columna': nombre_corto,
        'Mínimo': df_chile[col].min(),
        'Máximo': df_chile[col].max()
    })

print("\nDESPUÉS de corregir outliers:")
df_resumen_despues = pd.DataFrame(resumen_despues)
display(df_resumen_despues)

ANTES de corregir outliers:


,Columna,Mínimo,Máximo,Fuera de rango
0,retail_and_recreation,-97.0,133.0,0
1,grocery_and_pharmacy,-96.0,124.0,0
2,parks,-100.0,211.0,1
3,transit_stations,-100.0,322.0,20
4,workplaces,-100.0,146.0,0
5,residential,-6.0,47.0,0



DESPUÉS de corregir outliers:


,Columna,Mínimo,Máximo
0,retail_and_recreation,-97.0,133.0
1,grocery_and_pharmacy,-96.0,124.0
2,parks,-100.0,200.0
3,transit_stations,-100.0,200.0
4,workplaces,-100.0,146.0
5,residential,-6.0,47.0
